In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score

/opt/anaconda3/envs/tf_m1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the data
def load_real_data(filepath='creditcard.csv'):
    try:
        df = pd.read_csv(filepath)
        print(f'Dataset loaded successfully. Shape: {df.shape}')
        return df
    except FileNotFoundError:
        print(f"Error: '{filepath}' not found. Please download it from Kaggle.")
        return None

df = load_real_data()

Dataset loaded successfully. Shape: (284807, 31)


In [3]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [6]:
if df is not None:
    # Prepare features and target
    X = df.drop('Class', axis=1).values
    y = df['Class'].values

    # Optimized Objective function
    def objective(trial):
        # Hyperparameter Search Space
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6,1.0),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 50,150),
            'n_estimators':50,
            'eval_metric': 'aucpr',
            'use_label_encoder': False,
            'verbosity':0
        }

        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size = 0.2, stratify=y, random_state=42
        )

        model = xgb.XGBClassifier(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_pred = model.predict_proba(X_val)[:,1]

        ap = average_precision_score(y_val, y_pred)

        trial.report(ap, 0)
        if trial.should_prune():
            raise optuna.TrialPruned()
        return ap

    study = optuna.create_study(
        direction = 'maximize',
        sampler = optuna.samplers.TPESampler(seed=42),
        pruner= optuna.pruners.SuccessiveHalvingPruner()
    )

    study.optimize(objective, n_trials=20, n_jobs=-1)

[I 2026-08-08 11:03:55,815] A new study created in memory with name: no-name-d99427a7-9434-4140-b4b9-7d6ed42937a8
[I 2026-08-08 11:03:59,357] Trial 11 finished with value: 0.7137914686959793 and parameters: {'max_depth': 3, 'learning_rate': 0.09949206554545967, 'subsample': 0.7192417249640433, 'colsample_bytree': 0.8957088186036228, 'scale_pos_weight': 78.960182806899}. Best is trial 11 with value: 0.7137914686959793.
[I 2026-08-08 11:03:59,552] Trial 16 finished with value: 0.7044692283091978 and parameters: {'max_depth': 3, 'learning_rate': 0.03224073471758192, 'subsample': 0.8228317628797706, 'colsample_bytree': 0.7952475147311445, 'scale_pos_weight': 119.09639665863708}. Best is trial 11 with value: 0.7137914686959793.
[I 2026-08-08 11:03:59,810] Trial 0 finished with value: 0.8389920062498974 and parameters: {'max_depth': 5, 'learning_rate': 0.14114895678977485, 'subsample': 0.8706172714219946, 'colsample_bytree': 0.8993239907304508, 'scale_pos_weight': 129.58442217664123}. Best i